# Week 1: Data acquisition and audit (Group 2)
**Project:** Can We Trust AI-Image Detectors on Ugandan Content?

This notebook does Week 1 of the plan in `docs/data_guide`: stream a balanced subset of the public datasets from Hugging Face into our shared Google Drive, build one metadata table, audit it, and draft the split. **All six members run the same notebook with their own `MEMBER` code.** Each member pulls a different random slice (different seed) into their own metadata file, so the work adds up without clashing.

**Before you run it (once per member)**
1. Open the shared Drive folder `DEEPFAKE_CV_DATA` that Andrina shared, and use *Add shortcut to Drive* so it appears in **My Drive**.
2. In the first config cell set `MEMBER` to your own code (GW, AW, NG, NC, SM or PN).
3. Keep `PILOT = True` for your first run (a few hundred images). Only set it to `False` after the pilot works and the audit looks sensible.
4. Runtime > Change runtime type: CPU is enough for this notebook.

**Rules we agreed:** originals are saved byte for byte as they come from the source (no resizing, no re-saving). File names carry no label. Labels live only in the metadata table. Raw images stay in Drive, never in GitHub. Before you commit this notebook to GitHub, use *Edit > Clear all outputs*.

## 0. Setup

In [ ]:
%pip -q install -U datasets imagehash tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Hugging Face token (recommended, free)
Both datasets look public, so streaming may work without a token, but a free **read** token avoids rate limits and is needed if a dataset turns out to be gated.
1. Make a token at huggingface.co > Settings > Access Tokens. Choose **Read** (not Write). Use your own token, never someone else's.
2. In Colab, click the key icon (Secrets) on the left, add a secret named `HF_TOKEN`, paste the token, and switch on *Notebook access*.
3. Never paste a token into a cell or commit it to GitHub. This notebook reads it from Colab Secrets only.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('Hugging Face token loaded from Colab Secrets.')
except Exception as e:
    print('No HF_TOKEN secret found, continuing without one (public datasets may still work, but slower).')

## 1. Configuration (edit `MEMBER` and `PILOT` only)

In [ ]:
ROOT   = '/content/drive/MyDrive/DEEPFAKE_CV_DATA'   # the shared folder shortcut
MEMBER = 'GW'      # GW=Grace, AW=Andrina, NG=Gloria, NC=Charity, SM=Shane, PN=Priscilla
PILOT  = True      # True: a few hundred images to test. False: the real Week 1 pull.

SEEDS = {'GW': 11, 'AW': 22, 'NG': 33, 'NC': 44, 'SM': 55, 'PN': 66}
assert MEMBER in SEEDS, f'MEMBER must be one of {list(SEEDS)}'

# Images wanted PER CLASS (real and AI each), per member. Six members together give about
# 27 thousand public images in the full run, inside the planned 20 to 40 thousand.
if PILOT:
    N_PER_CLASS = {'openfake': 150, 'community': 100, 'reddit': 60}
    MY_BUDGET_GB, MAX_MINUTES = 0.4, 15
else:
    N_PER_CLASS = {'openfake': 1500, 'community': 500, 'reddit': 250}
    # Files you save into the shared folder normally count against YOUR Google Drive space.
    # Set this to less than your free space (Drive > Storage shows it). It is shared across the 3 sources.
    MY_BUDGET_GB, MAX_MINUTES = 1.5, 90
MAX_GB = MY_BUDGET_GB / 3                  # per source
print('Member:', MEMBER, '| pilot:', PILOT, '|', N_PER_CLASS, '| budget GB (all sources):', MY_BUDGET_GB)

## 2. Helpers: folders, source definitions, and the streaming puller
Each source is described once in `SPECS`. Real images and AI images are counted separately, and we cap how many images any one generator (or real-image source) can give, so no single generator dominates.

**What I could not check from outside Colab:** the column names of the OpenFake `reddit` subset. The code prints its columns on the first run. If they differ from `core`, fix the `SPECS['reddit']` lines and rerun.

In [ ]:
import io, os, re, time, json, glob, math, hashlib, random
from collections import Counter
import numpy as np, pandas as pd
from PIL import Image as PILImage

FOLDERS = ['public_data/openfake', 'public_data/community_forensics',
           'in_the_wild/openfake_reddit', 'in_the_wild/truefake',
           'kampala/originals', 'kampala/whatsapp_photo', 'kampala/whatsapp_document',
           'metadata', 'audit_outputs']

def make_folders(root):
    for f in FOLDERS:
        os.makedirs(os.path.join(root, f), exist_ok=True)

EXT = {'JPEG': 'jpg', 'MPO': 'jpg', 'PNG': 'png', 'WEBP': 'webp',
       'GIF': 'gif', 'BMP': 'bmp', 'TIFF': 'tif'}
META_COLS = ['image_id', 'path', 'label', 'source', 'source_detail', 'generator',
             'original_id', 'version', 'role', 'format', 'width', 'height', 'mode',
             'file_bytes', 'sha256', 'license', 'creator', 'pulled_at']

SPECS = {
  'openfake': dict(
      repo='ComplexDataLab/OpenFake', config='core', split='train',
      image_col='image', decode_feature=True, buffer=500,
      label_fn=lambda ex: {'real': 'real', 'fake': 'ai'}.get(str(ex.get('label')).lower()),
      gen_fn=lambda ex, lab: 'real' if lab == 'real' else str(ex.get('model')),
      detail_fn=lambda ex: str(ex.get('type')),         # image source or kind
      orig_fn=lambda ex: str(ex.get('row_id')),
      skip_fn=lambda ex: False,
      role='train_val', folder='public_data/openfake', prefix='OF',
      license='CC BY-NC 4.0', cap={'real': 300, 'ai': 40}),
  'community': dict(
      repo='OwensLab/CommunityForensics-Small', config=None, split='train',
      image_col='image_data', decode_feature=False, buffer=1000,
      label_fn=lambda ex: {0: 'real', 1: 'ai'}.get(int(ex.get('label'))),
      gen_fn=lambda ex, lab: 'real' if lab == 'real' else str(ex.get('model_name')),
      detail_fn=lambda ex: str(ex.get('real_source') if int(ex.get('label')) == 0
                               else ex.get('architecture')),
      orig_fn=lambda ex: str(ex.get('image_name')),
      skip_fn=lambda ex: bool(ex.get('nsfw_flag')),
      role='train_val', folder='public_data/community_forensics', prefix='CF',
      license='CC BY-NC-SA 4.0', cap={'real': 150, 'ai': 3}),
  'reddit': dict(
      repo='ComplexDataLab/OpenFake', config='reddit', split=None,   # split auto-detected
      image_col='image', decode_feature=True, buffer=500,
      label_fn=lambda ex: {'real': 'real', 'fake': 'ai'}.get(str(ex.get('label')).lower()),
      gen_fn=lambda ex, lab: 'real' if lab == 'real' else str(ex.get('model')),
      detail_fn=lambda ex: str(ex.get('type')),
      orig_fn=lambda ex: str(ex.get('row_id')),
      skip_fn=lambda ex: False,
      role='wild_test', folder='in_the_wild/openfake_reddit', prefix='OR',
      license='CC BY-NC 4.0', cap={'real': 300, 'ai': 60}),
}

def get_bytes(v):
    # Hugging Face gives {'bytes':..., 'path':...} (decode=False) or raw bytes.
    if isinstance(v, dict):
        v = v.get('bytes')
    if isinstance(v, (bytes, bytearray)):
        return bytes(v)
    raise ValueError(f'unsupported image value type: {type(v)}')

def group_of(label, generator, detail):
    # What the per-group cap applies to: the generator for AI, the real-image source for real.
    return generator if label == 'ai' else detail

def meta_path_for(root, name, member):
    return os.path.join(root, 'metadata', f'public_{name}_{member}.csv')

In [ ]:
def pull_from_stream(stream, name, member, root, n_per_class, max_bytes, max_seconds,
                     flush_every=100):
    # Take a balanced, generator-capped set from `stream` and save the ORIGINAL bytes.
    spec = SPECS[name]
    out_dir = os.path.join(root, spec['folder'])
    os.makedirs(out_dir, exist_ok=True)
    meta_path = meta_path_for(root, name, member)
    os.makedirs(os.path.dirname(meta_path), exist_ok=True)
    counts, per_group, seen, k, used_bytes = Counter(), Counter(), set(), 0, 0
    if os.path.exists(meta_path):                       # resume after a crash or timeout
        old = pd.read_csv(meta_path)
        for r in old.itertuples():
            counts[r.label] += 1
            per_group[(r.label, group_of(r.label, r.generator, r.source_detail))] += 1
            seen.add(r.sha256)
        k, used_bytes = len(old), int(old.file_bytes.sum())
        print(f'Resuming {name}: {len(old)} images already saved by {member}')
    buf, skipped, t0, first = [], Counter(), time.time(), True

    def flush():
        if buf:
            pd.DataFrame(buf, columns=META_COLS).to_csv(
                meta_path, mode='a', header=not os.path.exists(meta_path), index=False)
            buf.clear()

    for ex in stream:
        if first:
            print(f'[{name}] columns:', sorted(ex.keys())); first = False
        if time.time() - t0 > max_seconds:
            print(f'[{name}] time limit reached'); break
        if used_bytes > max_bytes:
            print(f'[{name}] storage limit reached'); break
        if all(counts[c] >= n_per_class for c in ('real', 'ai')):
            break
        label = spec['label_fn'](ex)
        if label is None or spec['skip_fn'](ex):
            skipped['label_or_flag'] += 1; continue
        if counts[label] >= n_per_class:
            continue
        gen, detail = spec['gen_fn'](ex, label), spec['detail_fn'](ex)
        g = (label, group_of(label, gen, detail))
        if per_group[g] >= spec['cap'][label]:
            skipped['group_cap'] += 1; continue
        try:
            raw = get_bytes(ex[spec['image_col']])
            im = PILImage.open(io.BytesIO(raw))          # header only, no full decode
            fmt, (w, h), mode = im.format, im.size, im.mode
        except Exception as e:
            skipped['unreadable'] += 1; continue
        sha = hashlib.sha256(raw).hexdigest()
        if sha in seen:
            skipped['duplicate'] += 1; continue
        k += 1
        image_id = f"{spec['prefix']}_{member}_{k:06d}"
        fname = f"{image_id}.{EXT.get(fmt, (fmt or 'bin').lower())}"
        with open(os.path.join(out_dir, fname), 'wb') as f:
            f.write(raw)
        seen.add(sha); counts[label] += 1; per_group[g] += 1; used_bytes += len(raw)
        buf.append(dict(image_id=image_id, path=os.path.join(spec['folder'], fname),
                        label=label, source=name, source_detail=detail, generator=gen,
                        original_id=spec['orig_fn'](ex), version='orig', role=spec['role'],
                        format=fmt, width=w, height=h, mode=mode, file_bytes=len(raw),
                        sha256=sha, license=spec['license'], creator=member,
                        pulled_at=time.strftime('%Y-%m-%d %H:%M:%S')))
        if len(buf) >= flush_every:
            flush(); print(f'[{name}] saved so far: {dict(counts)}')
    flush()
    return dict(source=name, member=member, counts=dict(counts), skipped=dict(skipped),
                gigabytes=round(used_bytes / 1e9, 3), seconds=round(time.time() - t0))

In [ ]:
def open_stream(name, seed):
    # Hugging Face streaming: nothing is downloaded up front.
    from datasets import load_dataset, get_dataset_split_names, Image as HFImage
    spec = SPECS[name]
    split = spec['split']
    if split is None:
        splits = get_dataset_split_names(spec['repo'], spec['config'])
        print(f'[{name}] available splits: {splits}')
        split = splits[0]
    ds = load_dataset(spec['repo'], name=spec['config'], split=split, streaming=True)
    if spec['decode_feature']:
        ds = ds.cast_column(spec['image_col'], HFImage(decode=False))   # keep the original bytes
    return ds.shuffle(seed=seed, buffer_size=spec['buffer'])

## 3. Create the folders and run your pull

In [ ]:
make_folders(ROOT)
RUN = ['openfake', 'community', 'reddit']       # remove a name to skip that source
results = []
for name in RUN:
    stream = open_stream(name, SEEDS[MEMBER])
    res = pull_from_stream(stream, name, MEMBER, ROOT, N_PER_CLASS[name],
                           max_bytes=MAX_GB * 1e9, max_seconds=MAX_MINUTES * 60)
    print(res); results.append(res)

**Check before you scale up (the pilot):** each source shows both `real` and `ai` counts close to the target; the printed columns look right; no `unreadable` explosion. If OpenFake `reddit` printed different column names, fix `SPECS['reddit']` first. Then set `PILOT = False` and run again. The code resumes from where it stopped, so re-running is safe.

**Storage warning:** a full run is large. Files you save into the shared folder normally count against your own Google Drive space (free accounts have 15 GB in total, shared with Gmail and Photos), so set `MY_BUDGET_GB` below your free space. Check the `gigabytes` figure after the pilot. If six members together will not fit, lower `N_PER_CLASS` or ask the owner of the folder for more space.

## 4. Audit (run after at least a few members have pulled)
Merges every member's public metadata and produces the Week 1 evidence: counts, formats, sizes, corrupt files, duplicates, shortcut check, and a sample grid. Outputs go to `audit_outputs/`.

In [ ]:
def load_public_meta(root):
    files = sorted(glob.glob(os.path.join(root, 'metadata', 'public_*.csv')))
    assert files, 'No public_*.csv metadata found yet. Run the pull first.'
    meta = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    return meta

meta = load_public_meta(ROOT)
OUT = os.path.join(ROOT, 'audit_outputs'); os.makedirs(OUT, exist_ok=True)

counts_class_source = (meta.groupby(['role', 'source', 'label']).size()
                       .unstack(fill_value=0).assign(total=lambda d: d.sum(axis=1)))
counts_by_member = meta.groupby(['creator', 'source', 'label']).size().unstack(fill_value=0)
counts_by_generator = (meta[meta.label == 'ai'].groupby(['source', 'generator']).size()
                       .rename('n').reset_index().sort_values('n', ascending=False))
print(counts_class_source, '\n'); print(counts_by_member, '\n')
print('Distinct AI generators per source:\n', counts_by_generator.groupby('source').generator.nunique())
counts_class_source.to_csv(f'{OUT}/counts_by_class_source.csv')
counts_by_member.to_csv(f'{OUT}/counts_by_member.csv')
counts_by_generator.to_csv(f'{OUT}/counts_by_generator.csv', index=False)

In [ ]:
# Image properties and the SHORTCUT CHECK: can the model tell real from AI without looking at content?
meta['megapixels'] = meta.width * meta.height / 1e6
meta['bytes_per_pixel'] = meta.file_bytes / (meta.width * meta.height)

format_table = (meta.groupby(['source', 'label', 'format']).size()
                .unstack(fill_value=0))
prop_table = (meta.groupby(['source', 'label'])
              [['width', 'height', 'megapixels', 'file_bytes', 'bytes_per_pixel']]
              .median().round(3))
print('Format by source and class:\n', format_table, '\n')
print('Median size and file size by source and class:\n', prop_table, '\n')

# Flag any source where the share of JPEG differs a lot between real and AI
flags = []
for src, d in meta.groupby('source'):
    jp = d.assign(is_jpeg=d.format.isin(['JPEG', 'MPO'])).groupby('label').is_jpeg.mean()
    if {'real', 'ai'} <= set(jp.index) and abs(jp['real'] - jp['ai']) > 0.30:
        flags.append(f"{src}: JPEG share real={jp['real']:.0%} vs ai={jp['ai']:.0%}")
    med = d.groupby('label').megapixels.median()
    if {'real', 'ai'} <= set(med.index) and max(med) / max(min(med), 1e-9) > 2:
        flags.append(f"{src}: median megapixels real={med['real']:.2f} vs ai={med['ai']:.2f}")
print('SHORTCUT FLAGS:', flags if flags else 'none above thresholds')
format_table.to_csv(f'{OUT}/format_by_source_class.csv'); prop_table.to_csv(f'{OUT}/properties_by_source_class.csv')
with open(f'{OUT}/shortcut_flags.txt', 'w') as f:
    f.write('\n'.join(flags) if flags else 'none above thresholds')

In [ ]:
# Corrupt files and exact duplicates (cheap), then perceptual duplicates (slower, cached).
def check_readable(root, rel):
    try:
        with PILImage.open(os.path.join(root, rel)) as im:
            im.load()
        return True
    except Exception:
        return False

import imagehash
cache_path = f'{OUT}/phash_cache.csv'
cache = pd.read_csv(cache_path) if os.path.exists(cache_path) else pd.DataFrame(columns=['image_id', 'phash', 'ok'])
todo = meta[~meta.image_id.isin(cache.image_id)]
PHASH_BATCH = 4000          # per run, so a Colab timeout never loses work
rows = []
for r in todo.head(PHASH_BATCH).itertuples():
    try:
        with PILImage.open(os.path.join(ROOT, r.path)) as im:
            im.load(); h = str(imagehash.phash(im)); ok = True
    except Exception:
        h, ok = '', False
    rows.append(dict(image_id=r.image_id, phash=h, ok=ok))
cache = pd.concat([cache, pd.DataFrame(rows)], ignore_index=True)
cache.to_csv(cache_path, index=False)
print(f'hashed {len(cache)} of {len(meta)} images. Re-run this cell until all are hashed.')

corrupt = meta.merge(cache[cache.ok == False], on='image_id')[['image_id', 'path']]
corrupt.to_csv(f'{OUT}/corrupt_files.csv', index=False)
print('unreadable images:', len(corrupt))

exact = meta[meta.duplicated('sha256', keep=False)].sort_values('sha256')
print('exact duplicate rows (same bytes):', len(exact))
exact[['sha256', 'image_id', 'source', 'role', 'label']].to_csv(f'{OUT}/exact_duplicates.csv', index=False)

In [ ]:
# Near-duplicates by perceptual hash (hamming distance <= 4). Flags pairs, including pairs
# that cross train and test sources, which is the leak we most want to catch.
def popcount64(x):
    return np.bitwise_count(x) if hasattr(np, 'bitwise_count') else \
        np.array([bin(int(v)).count('1') for v in x])

good = cache[(cache.ok == True) & (cache.phash != '')].merge(meta[['image_id', 'role', 'source', 'label']], on='image_id')
H = np.array([int(h, 16) for h in good.phash], dtype=np.uint64)
pairs = []
for i in range(len(H)):
    d = popcount64(H[i + 1:] ^ H[i])
    for j in np.nonzero(d <= 4)[0]:
        pairs.append((good.image_id.iat[i], good.image_id.iat[i + 1 + j]))
near = pd.DataFrame(pairs, columns=['a', 'b'])
if len(near):
    near = (near.merge(meta[['image_id', 'role', 'source', 'label']].add_suffix('_a'), left_on='a', right_on='image_id_a')
                .merge(meta[['image_id', 'role', 'source', 'label']].add_suffix('_b'), left_on='b', right_on='image_id_b'))
    cross = near[near.role_a != near.role_b]
    print('near-duplicate pairs:', len(near), '| across train_val and wild_test:', len(cross))
    near.to_csv(f'{OUT}/near_duplicates.csv', index=False)
else:
    print('no near-duplicate pairs found in the hashed images')

In [ ]:
# Sample grid for the report (Figure 1): 4 real + 4 AI per source. Public data only.
import matplotlib; import matplotlib.pyplot as plt
rng = random.Random(7)
srcs = [s for s in ['openfake', 'community', 'reddit'] if s in set(meta.source)]
fig, axes = plt.subplots(2 * len(srcs), 4, figsize=(12, 6 * len(srcs)))
axes = np.atleast_2d(axes)
for si, s in enumerate(srcs):
    for li, lab in enumerate(['real', 'ai']):
        pool = meta[(meta.source == s) & (meta.label == lab)]
        pick = pool.sample(min(4, len(pool)), random_state=rng.randint(0, 10**6)) if len(pool) else pool
        for ci in range(4):
            ax = axes[2 * si + li, ci]; ax.axis('off')
            if ci < len(pick):
                r = pick.iloc[ci]
                with PILImage.open(os.path.join(ROOT, r.path)) as im:
                    ax.imshow(im.convert('RGB'))
                ax.set_title(f'{lab} | {r.generator if lab == "ai" else r.source_detail}', fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT}/sample_grid_draft.png', dpi=150); plt.show()
print('Saved draft grid. Before it goes in the report, replace any image that shows an identifiable '
      'person or a copyrighted meme/logo, and keep long labels short.')

## 5. Draft split (Step 6 of the guide): held-out generators first, before any training
Rules: an image and all its copies stay in the same part; any in-the-wild image that duplicates a training-source image is dropped (`dropped_leak`); near and exact duplicates stay together; a set of AI generators is held out so they appear **only** in `test_unseen`; the in-the-wild images are test only. Write down the rules and counts now. Exact numbers are final only after the audit is clean.

In [ ]:
SPLIT_SEED, HELDOUT_FRACTION, MIN_IMAGES_PER_HELDOUT_GEN = 2026, 0.20, 5

def draft_split(meta, near=None, seed=SPLIT_SEED):
    rng = np.random.RandomState(seed)
    m = meta.copy()
    m['split'] = ''
    m.loc[m.role == 'wild_test', 'split'] = 'wild_test'
    m.loc[m.role == 'kampala_locked', 'split'] = 'kampala_locked'
    tv = m[m.role == 'train_val']

    # 1) choose held-out generators among AI generators with enough images
    gen_n = tv[tv.label == 'ai'].groupby('generator').size()
    cand = sorted(gen_n[gen_n >= MIN_IMAGES_PER_HELDOUT_GEN].index)
    k = max(3, int(round(HELDOUT_FRACTION * len(cand)))) if len(cand) >= 3 else len(cand)
    heldout = sorted(rng.choice(cand, size=min(k, len(cand)), replace=False).tolist()) if cand else []

    # 2) group ids: copies of the same image (same sha256, or linked by a near-duplicate pair) stay together
    parent = {i: i for i in tv.image_id}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b):
        if a in parent and b in parent:
            parent[find(a)] = find(b)
    for _, d in tv.groupby('sha256'):
        ids = d.image_id.tolist()
        for x in ids[1:]: union(ids[0], x)
    if near is not None and len(near):
        for a, b in zip(near['a'], near['b']): union(a, b)
    tv = tv.assign(group=[find(i) for i in tv.image_id])

    # 3) a group that contains a held-out generator image goes wholly to test_unseen
    held_groups = set(tv[tv.generator.isin(heldout)].group)
    tv.loc[tv.group.isin(held_groups), 'split'] = 'test_unseen'
    # real images: set aside a share as the real side of test_unseen
    rest = tv[tv.split == '']
    for lab, share in (('real', 0.10),):
        gs = sorted(set(rest[rest.label == lab].group)); rng.shuffle(gs)
        cut = set(gs[: int(share * len(gs))]); tv.loc[tv.group.isin(cut) & (tv.split == ''), 'split'] = 'test_unseen'
    # 4) the rest: 80 percent train, 10 validation, 10 test_seen, by group
    rest = tv[tv.split == '']
    gs = sorted(set(rest.group)); rng.shuffle(gs)
    n = len(gs); a, b = int(0.8 * n), int(0.9 * n)
    assign = {g: ('train' if i < a else 'val' if i < b else 'test_seen') for i, g in enumerate(gs)}
    tv.loc[tv.split == '', 'split'] = tv.loc[tv.split == '', 'group'].map(assign)

    # in-the-wild images that duplicate a training-source image (same bytes or near-duplicate) are dropped
    tv_sha = set(m[m.role == 'train_val'].sha256)
    m.loc[(m.role == 'wild_test') & m.sha256.isin(tv_sha), 'split'] = 'dropped_leak'
    if near is not None and len(near):
        role_of = dict(zip(m.image_id, m.role))
        for a_, b_ in zip(near['a'], near['b']):
            ra, rb = role_of.get(a_), role_of.get(b_)
            if {ra, rb} == {'train_val', 'wild_test'}:
                w = a_ if ra == 'wild_test' else b_
                m.loc[m.image_id == w, 'split'] = 'dropped_leak'
    m = m.merge(tv[['image_id', 'split', 'group']].rename(columns={'split': 'split_tv'}), on='image_id', how='left')
    m['split'] = np.where(m.role == 'train_val', m.split_tv, m.split)
    m = m.drop(columns=['split_tv'])
    # safety checks: no group or sha256 in two splits
    chk = m[m.role == 'train_val']
    assert not ((m.role == 'wild_test') & m.sha256.isin(tv_sha) & (m.split != 'dropped_leak')).any()
    assert chk.groupby('group').split.nunique().max() == 1, 'a duplicate group crosses splits'
    assert chk.groupby('sha256').split.nunique().max() == 1, 'identical bytes cross splits'
    leak_gen = set(chk[chk.split == 'train'].generator) & set(heldout)
    assert not leak_gen, f'held-out generators found in train: {leak_gen}'
    return m, heldout

near_df = near if 'near' in globals() and isinstance(near, pd.DataFrame) and len(near) else None
split_df, heldout = draft_split(meta, near_df)
print('Held-out generators:', heldout)
print(split_df.groupby(['split', 'label']).size().unstack(fill_value=0))
split_df.to_csv(os.path.join(ROOT, 'metadata', 'splits_draft.csv'), index=False)
with open(os.path.join(ROOT, 'metadata', 'heldout_generators.txt'), 'w') as f:
    f.write('\n'.join(heldout))

## 6. Kampala pilot set: check the shared log (counts and file existence only)
Everyone saves their Kampala files and fills the shared log (template: `kampala_log_template.xlsx`). Export the sheet as CSV to `metadata/kampala_log.csv`. This cell **never opens or shows the images**. It only checks that the log is complete and the files exist. Do not print or commit Kampala images.

In [ ]:
KAMPALA_DIRS = {'orig': 'kampala/originals', 'wa_photo': 'kampala/whatsapp_photo', 'wa_doc': 'kampala/whatsapp_document'}
REQUIRED = ['image_id', 'version', 'file_name', 'label', 'creator', 'device_or_tool',
            'contains_person', 'consent_obtained']

def check_kampala_log(log, root):
    problems = []
    miss = [c for c in REQUIRED if c not in log.columns]
    if miss: return [f'missing columns: {miss}'], None
    for c in REQUIRED:
        n = log[c].isna().sum() + (log[c].astype(str).str.strip() == '').sum()
        if n: problems.append(f'{n} blank values in {c}')
    log = log.copy()
    for img, d in log.groupby('image_id'):
        if set(d.version) != set(KAMPALA_DIRS):
            problems.append(f'{img}: needs exactly the 3 versions, has {sorted(set(d.version))}')
        if d.label.nunique() > 1: problems.append(f'{img}: label differs between versions')
        if (d.label == 'ai').any() and d.device_or_tool.isna().any():
            problems.append(f'{img}: AI image without generator name')
    for r in log.itertuples():
        if r.version in KAMPALA_DIRS and not os.path.exists(os.path.join(root, KAMPALA_DIRS[r.version], str(r.file_name))):
            problems.append(f'file not found: {KAMPALA_DIRS[r.version]}/{r.file_name}')
        if str(r.contains_person).lower() == 'yes' and str(r.consent_obtained).lower() != 'yes':
            problems.append(f'{r.image_id}: person shown but no consent recorded')
    summary = (log[log.version == 'orig'].groupby(['creator', 'label']).size().unstack(fill_value=0))
    return problems, summary

kpath = os.path.join(ROOT, 'metadata', 'kampala_log.csv')
if os.path.exists(kpath):
    problems, summary = check_kampala_log(pd.read_csv(kpath), ROOT)
    print('Originals per member and class (target: 150 to 200 real and 150 to 200 AI in total):\n', summary)
    print('\nPROBLEMS:' if problems else '\nNo problems found.')
    for p in problems[:50]: print(' -', p)
else:
    print('kampala_log.csv not exported yet. That is fine on day 1.')

## 7. Week 1 checklist (tick off in the group chat)
- [ ] Licences and access notes recorded for OpenFake, Community Forensics-Small, OpenFake Reddit (TrueFake access still to check by the owner of that task)
- [ ] Each member's pilot ran; then each member's full pull ran
- [ ] `audit_outputs/` has counts, format and size tables, shortcut flags, corrupt and duplicate reports
- [ ] Sample grid reviewed by a person, identifiable people and copyrighted items replaced
- [ ] Held-out generators written down in `metadata/heldout_generators.txt` before any training
- [ ] `metadata/splits_draft.csv` and the split rules saved
- [ ] Kampala log complete for every member, consent recorded where a person appears, three versions per image
- [ ] Short risk and open-issues list added to the README
- [ ] Every member can explain what each audit table shows